# VLM-Anomaly — Full MVTec Sweep · Claude Opus 4.7

**Model:** `claude-opus-4-7`  
**Strategy:** 10 images per API call (batched) to maximise throughput  
**Free-tier Opus limits:** 50 RPM · 500K input TPM · 80K output TPM  

| Scope | Images | Est. cost |
|---|---|---|
| 1 category (smoke) | ~83 | ~$2.70 |
| Full sweep (15 cat) | ~1,245 | ~$26–32 |

> **$5 credit covers ~2 categories.** Add more credit before running the full sweep.

In [ ]:
# ── Cell 1: Setup paths & sys.path ─────────────────────────────────────────
import sys
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT   = Path().resolve().parent
SRC_DIR     = REPO_ROOT / 'src'
PROMPTS_DIR = REPO_ROOT / 'prompts'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SRC_DIR.exists(),     f'src/ not found at {SRC_DIR}'
assert PROMPTS_DIR.exists(), f'prompts/ not found at {PROMPTS_DIR}'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

import vlm_anomaly
print(f'vlm_anomaly {vlm_anomaly.__version__} ready')
print(f'Results  : {RESULTS_DIR}')

In [ ]:
# ── Cell 2: Verify API key ───────────────────────────────────────────────────
import os

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
assert api_key, 'ANTHROPIC_API_KEY not set — add it to your .env file'
print(f'ANTHROPIC_API_KEY: {api_key[:12]}...{api_key[-4:]}')

In [ ]:
# ── Cell 3: Find MVTec dataset ──────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
]:
    if candidate.exists() and any(candidate.iterdir()):
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, f'MVTec not found. Expected at {REPO_ROOT}/data/mvtec'
categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {len(categories)} → {categories}')

In [ ]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
MODEL      = 'claude-opus-4-7'
PROMPT_KEY = 'manufacturing.detailed'
BATCH_SIZE = 10      # images per API call — max 20, 10 is reliable
BUDGET_USD = 5.0     # hard cap — raise after adding more credit

est_images = len(categories) * 83
est_cost   = est_images * 0.028   # ~$28 / 1M input tokens for Opus
print(f'Model        : {MODEL}')
print(f'Batch size   : {BATCH_SIZE} images/call')
print(f'Budget cap   : ${BUDGET_USD}  ← raise this when adding more credit')
print(f'Est. images  : ~{est_images}')
print(f'Est. cost    : ~${est_cost:.2f} full sweep')
print(f'  $5 covers  : ~{int(5 / (est_cost / est_images))} images (~{int(5 / (est_cost / est_images) / 83)} categories)')

In [ ]:
# ── Cell 5: SMOKE TEST — 1 image, verify API before full sweep ──────────────
from vlm_anomaly.backends.anthropic_backend import AnthropicBackend
from vlm_anomaly.logging import configure_logging

configure_logging(json_logs=False, log_level='INFO')
backend = AnthropicBackend(model=MODEL)

smoke_img = next((MVTEC_ROOT / 'bottle' / 'test').rglob('*.png'))
print(f'Smoke test image: {smoke_img.relative_to(REPO_ROOT)}')

smoke_result = backend.predict(
    smoke_img,
    'Is there any defect or anomaly? Reply with JSON only: '
    '{"is_anomalous": bool, "confidence": float, "defect_type": str, "description": str}'
)

print(f'  is_anomalous : {smoke_result.is_anomalous}')
print(f'  confidence   : {smoke_result.confidence:.2f}')
print(f'  defect_type  : {smoke_result.defect_type}')
print(f'  description  : {smoke_result.description[:100]}')
print(f'  latency_ms   : {smoke_result.latency_ms:.0f}')
print(f'  cost_usd     : ${smoke_result.cost_usd:.6f}')
print(f'  tokens_in    : {smoke_result.tokens_in}')
print(f'  tokens_out   : {smoke_result.tokens_out}')
print(f'  parse_error  : {smoke_result.parse_error}')
print()
print('✓ Smoke test passed — API key and model are working.')

In [ ]:
# ── Cell 6: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.evaluators.prompt_library import PromptLibrary

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
    default_budget_usd=BUDGET_USD,
)
settings.results_dir = RESULTS_DIR

dataset    = MVTec(root_dir=MVTEC_ROOT)
prompt_lib = PromptLibrary(prompts_dir=PROMPTS_DIR)
prompt_str = prompt_lib.render(PROMPT_KEY)

print(f'Dataset  : {MVTEC_ROOT}')
print(f'Backend  : {backend.name} / {MODEL}')
print(f'Prompt   : {PROMPT_KEY} ({len(prompt_str)} chars)')

In [ ]:
# ── Cell 7: Run all categories (batched, idempotent) ────────────────────────
import json
import time
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, f1_score

total_cost   = 0.0
all_category_results = []

for category in tqdm(categories, desc='MVTec categories'):
    out_file = RESULTS_DIR / f'anthropic_{MODEL.replace("/","-")}_mvtec_{category}.jsonl'
    if out_file.exists() and out_file.stat().st_size > 100:
        print(f'  [skip] {category} — already done')
        continue

    samples = dataset.get_test_samples(category)
    images  = [s.image_path for s in samples]
    labels  = [int(s.is_anomalous) for s in samples]

    predictions = []
    cat_cost    = 0.0

    # Process in batches of BATCH_SIZE
    for i in tqdm(range(0, len(images), BATCH_SIZE), desc=category, leave=False):
        batch_imgs = images[i : i + BATCH_SIZE]

        # Budget guard
        if total_cost + cat_cost >= BUDGET_USD:
            print(f'  [BUDGET] ${BUDGET_USD} cap reached — stopping.')
            break

        results = backend.predict_batch(batch_imgs, prompt_str)
        predictions.extend(results)
        cat_cost += sum(r.cost_usd for r in results)

    if not predictions:
        continue

    # Save per-image predictions
    with out_file.open('w') as f:
        for pred, label in zip(predictions, labels[:len(predictions)]):
            f.write(json.dumps({
                'image_path': str(pred.image_path),
                'is_anomalous': pred.is_anomalous,
                'confidence': pred.confidence,
                'defect_type': pred.defect_type,
                'description': pred.description,
                'ground_truth': bool(label),
                'latency_ms': pred.latency_ms,
                'cost_usd': pred.cost_usd,
                'tokens_in': pred.tokens_in,
                'tokens_out': pred.tokens_out,
                'parse_error': pred.parse_error,
                'model': MODEL,
                'category': category,
                'dataset': 'mvtec',
            }) + '\n')

    # Compute metrics
    gt    = labels[:len(predictions)]
    scores = [p.confidence if p.is_anomalous else 1 - p.confidence for p in predictions]
    preds  = [int(p.is_anomalous) for p in predictions]
    try:
        auroc = roc_auc_score(gt, scores)
    except Exception:
        auroc = float('nan')
    f1 = f1_score(gt, preds, zero_division=0)

    total_cost += cat_cost
    all_category_results.append({'category': category, 'auroc': auroc, 'f1': f1,
                                  'n': len(predictions), 'cost': cat_cost})
    print(f'  {category:12s}  AUROC={auroc:.3f}  F1={f1:.3f}  '
          f'n={len(predictions)}  cost=${cat_cost:.4f}')

print(f'\nDone. Total cost: ${total_cost:.4f}')

In [ ]:
# ── Cell 8: Summary table ────────────────────────────────────────────────────
import pandas as pd

if all_category_results:
    df = pd.DataFrame(all_category_results)
    print(f'Mean AUROC : {df.auroc.mean():.3f}')
    print(f'Mean F1    : {df.f1.mean():.3f}')
    print(f'Total cost : ${df.cost.sum():.4f}')
    print()
    display(df.sort_values('auroc', ascending=False).reset_index(drop=True))
else:
    print('No results yet — run Cell 7 first.')

In [ ]:
# ── Cell 9: Show result files + commit hint ──────────────────────────────────
result_files = sorted(RESULTS_DIR.glob('*anthropic*mvtec*.jsonl'))
print(f'Result files ({len(result_files)}):')
for f in result_files:
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')

print()
print('To commit:')
print('  git add results/*.jsonl')
print('  git commit -m "results(claude-opus-4-7): MVTec sweep via Anthropic API"')